In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

In [2]:
groundtruth = pd.read_csv("Noise_GroundTruth_and_TrainingLoss.csv")

foif = pd.read_csv("NoisyLabel_FOIF_Scores.csv")

tracin = pd.read_csv("NoisyLabel_TracIn_Scores.csv")

In [3]:
foif = foif.rename(columns={"Score":"FOIF_Score"})
tracin = tracin.rename(columns={"Score":"TracIn_Score"})

merged = (
    groundtruth
    .merge(
        foif[["Train_ID","FOIF_Score"]],
        on="Train_ID"
    )
    .merge(
        tracin[["Train_ID","TracIn_Score"]],
        on="Train_ID"
    )
)

print(merged.head())

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  FOIF_Score  \
0         1            1            0         1       1.639773   -0.798092   
1         2            1            1         0       0.560392    0.224097   
2         3            0            0         0       0.403948    0.153661   
3         4            1            1         0       0.175827    0.206177   
4         5            0            0         0       0.226923    0.039817   

   TracIn_Score  
0      3.234391  
1     -0.008772  
2      3.306922  
3     -0.002540  
4      3.411973  


In [4]:
random_baseline_seed = 42

rng = np.random.default_rng(random_baseline_seed)

merged["Random_Score"] = rng.random(len(merged))
# merged[["Train_ID", "Random_Score"]].to_csv(
#     "Random_Ranking.csv",
#     index=False
# )
print(merged)

       Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  \
0             1            1            0         1       1.639773   
1             2            1            1         0       0.560392   
2             3            0            0         0       0.403948   
3             4            1            1         0       0.175827   
4             5            0            0         0       0.226923   
...         ...          ...          ...       ...            ...   
53435     53436            0            0         0       0.192641   
53436     53437            0            0         0       0.192519   
53437     53438            0            0         0       0.197001   
53438     53439            0            0         0       0.350650   
53439     53440            1            1         0       0.176145   

       FOIF_Score  TracIn_Score  Random_Score  
0       -0.798092      3.234391      0.773956  
1        0.224097     -0.008772      0.438878  
2        0.1536

In [5]:
print(
    merged.groupby("is_noisy")[
        "FOIF_Score"
    ].mean()
)

print(
    merged.groupby("is_noisy")[
        "TracIn_Score"
    ].mean()
)

is_noisy
0    0.160337
1   -0.595723
Name: FOIF_Score, dtype: float64
is_noisy
0    1.697287
1    1.613734
Name: TracIn_Score, dtype: float64


In [6]:
def precision_recall_at_k(
    y_true,
    scores,
    k_percentage,
    retrieve="top"
):
    """
    Parameters
    ----------
    y_true : binary ground truth
    scores : ranking scores
    k_percentage : e.g. 0.1 for Top/Bottom 10%
    retrieve : "top" or "bottom"
    """

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = int(len(scores) * k_percentage)

    if retrieve == "top":
        selected = np.argsort(scores)[::-1][:k]

    elif retrieve == "bottom":
        selected = np.argsort(scores)[:k]

    else:
        raise ValueError("retrieve must be 'top' or 'bottom'")

    tp = y_true[selected].sum()

    precision = tp / k
    recall = tp / y_true.sum()

    return precision, recall

In [7]:
results = []

methods = {

    "Random":
    {
        "score": merged["Random_Score"],
        "retrieve": "bottom"
    },

    "Training Loss":
    {
        "score": merged["Training_Loss"],
        "retrieve": "top"
    },

    "FOIF":
    {
        "score": merged["FOIF_Score"],
        "retrieve": "bottom"
    },

    "TracIn":
    {
        "score": merged["TracIn_Score"],
        "retrieve": "bottom"
    }
}

In [8]:
for method, info in methods.items():

    scores = info["score"]

    retrieve = info["retrieve"]

    # AUPRC
    #
    # sklearn assumes larger score = positive
    #
    # Therefore we only negate FOIF/TracIn
    if retrieve == "bottom":
        auprc = average_precision_score(
            merged["is_noisy"],
            -scores
        )
    else:
        auprc = average_precision_score(
            merged["is_noisy"],
            scores
        )

    row = {

        "Method": method,

        "AUPRC": auprc
    }

    for p in [0.10,0.20,0.30]:

        precision, recall = precision_recall_at_k(

            merged["is_noisy"],

            scores,

            p,

            retrieve
        )

        row[f"Precision@{int(p*100)}"] = precision

        row[f"Recall@{int(p*100)}"] = recall

    results.append(row)

In [9]:
result_df = pd.DataFrame(results)

print(result_df.round(4))

          Method   AUPRC  Precision@10  Recall@10  Precision@20  Recall@20  \
0         Random  0.1977        0.1903     0.0952        0.1955     0.1955   
1  Training Loss  0.9795        0.9991     0.4995        0.9220     0.9220   
2           FOIF  0.9636        0.9951     0.4976        0.9162     0.9162   
3         TracIn  0.6065        0.9173     0.4586        0.4976     0.4976   

   Precision@30  Recall@30  
0        0.1984     0.2975  
1        0.6632     0.9948  
2        0.6466     0.9699  
3        0.3322     0.4983  
